In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# --- Paths ---
DATA_DIR = "/kaggle/input/competitions/titanic"
OUTPUT_PATH = "/kaggle/working/submission.csv"

# --- Load data ---
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
baseline = pd.read_csv(f"{DATA_DIR}/gender_submission.csv")


# --- Feature engineering ---
def add_features(df):
    df = df.copy()
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
    rare = ["Dr", "Rev", "Col", "Major", "Capt", "Sir", "Lady", "Don",
            "Dona", "Jonkheer", "the Countess"]
    df["Title"] = df["Title"].replace(rare, "Rare")
    df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["HasCabin"] = df["Cabin"].notna().astype(int)
    df["Deck"] = df["Cabin"].str[0].fillna("U")
    return df


train = add_features(train)
test = add_features(test)

numeric_features = ["Age", "Fare", "SibSp", "Parch", "FamilySize", "IsAlone", "HasCabin"]
categorical_features = ["Pclass", "Sex", "Embarked", "Title", "Deck"]

X = train[numeric_features + categorical_features]
y = train["Survived"]
X_test = test[numeric_features + categorical_features]

# --- Preprocessing + model ---
preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

model = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=500, max_depth=6, min_samples_leaf=2,
        random_state=42, n_jobs=-1,
    )),
])

# --- Cross-validation (printed) ---
scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print("=" * 50)
print("CROSS-VALIDATION RESULTS")
print("=" * 50)
for i, s in enumerate(scores, 1):
    print(f"Fold {i}: {s:.4f}")
print(f"Mean accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

# --- Train on full data and predict ---
model.fit(X, y)
preds = model.predict(X_test).astype(int)

# --- Top feature importances (printed) ---
feature_names = model.named_steps["prep"].get_feature_names_out()
importances = model.named_steps["clf"].feature_importances_
top = (pd.Series(importances, index=feature_names)
         .sort_values(ascending=False).head(10))
print("\n" + "=" * 50)
print("TOP 10 FEATURE IMPORTANCES")
print("=" * 50)
print(top.round(4).to_string())

# --- Prediction summary (printed) ---
submission = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": preds})

# Compare with the gender_submission baseline (aligned on PassengerId)
merged = submission.merge(baseline, on="PassengerId", suffixes=("_model", "_baseline"))
agreement = (merged["Survived_model"] == merged["Survived_baseline"]).mean()

print("\n" + "=" * 50)
print("PREDICTION SUMMARY")
print("=" * 50)
print(f"Shape: {submission.shape}  (expected: (418, 2))")
print(f"Predicted survived: {preds.sum()} ({preds.mean():.1%})")
print(f"Predicted deceased: {(preds == 0).sum()} ({1 - preds.mean():.1%})")
print(f"Agreement with gender_submission baseline: {agreement:.1%}")
print("\nFirst 10 predictions:")
print(submission.head(10).to_string(index=False))

# --- Submission file (required by the competition format) ---
submission.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved submission to {OUTPUT_PATH}")

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_val_score, cross_val_predict,
                                     RandomizedSearchCV, learning_curve)
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report,
                             roc_curve, auc)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              AdaBoostClassifier, GradientBoostingClassifier,
                              HistGradientBoostingClassifier,
                              VotingClassifier, StackingClassifier)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
SEED = 42

# ============================================================
# 1. LOAD DATA
# ============================================================
DATA_DIR = "/kaggle/input/competitions/titanic"
OUTPUT_PATH = "/kaggle/working/submission.csv"

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
baseline = pd.read_csv(f"{DATA_DIR}/gender_submission.csv")

# ============================================================
# 2. FEATURE ENGINEERING
# ============================================================
ticket_counts = pd.concat([train["Ticket"], test["Ticket"]]).value_counts()


def add_features(df):
    df = df.copy()
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
    rare = ["Dr", "Rev", "Col", "Major", "Capt", "Sir", "Lady", "Don",
            "Dona", "Jonkheer", "the Countess"]
    df["Title"] = df["Title"].replace(rare, "Rare")
    df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["HasCabin"] = df["Cabin"].notna().astype(int)
    df["Deck"] = df["Cabin"].str[0].fillna("U")
    df["TicketGroupSize"] = df["Ticket"].map(ticket_counts)
    df["LogFare"] = np.log1p(df["Fare"])
    return df


train = add_features(train)
test = add_features(test)

numeric_features = ["Age", "LogFare", "SibSp", "Parch", "FamilySize",
                    "IsAlone", "HasCabin", "TicketGroupSize"]
categorical_features = ["Pclass", "Sex", "Embarked", "Title", "Deck"]

X = train[numeric_features + categorical_features]
y = train["Survived"]
X_test = test[numeric_features + categorical_features]

# ============================================================
# 3. EXPLORATORY ANALYSIS (printed + plots)
# ============================================================
print("=" * 60)
print("DATA OVERVIEW")
print("=" * 60)
print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Overall survival rate: {y.mean():.1%}")
print("\nMissing values (train):")
print(train.isna().sum()[lambda s: s > 0].to_string())
print("\nSurvival rate by Sex:")
print(train.groupby("Sex")["Survived"].mean().round(3).to_string())
print("\nSurvival rate by Pclass:")
print(train.groupby("Pclass")["Survived"].mean().round(3).to_string())
print("\nSurvival rate by Title:")
print(train.groupby("Title")["Survived"].agg(["mean", "count"]).round(3).to_string())

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle("Titanic - Exploratory Data Analysis", fontsize=18, y=1.01)

sns.countplot(data=train, x="Sex", hue="Survived", ax=axes[0, 0])
axes[0, 0].set_title("Survival by Sex")

sns.countplot(data=train, x="Pclass", hue="Survived", ax=axes[0, 1])
axes[0, 1].set_title("Survival by Class")

sns.countplot(data=train, x="Embarked", hue="Survived", ax=axes[0, 2])
axes[0, 2].set_title("Survival by Port")

sns.countplot(data=train, x="Title", hue="Survived", ax=axes[0, 3])
axes[0, 3].set_title("Survival by Title")
axes[0, 3].tick_params(axis="x", rotation=45)

sns.histplot(data=train, x="Age", hue="Survived", kde=True, bins=30, ax=axes[1, 0])
axes[1, 0].set_title("Age Distribution")

sns.histplot(data=train, x="LogFare", hue="Survived", kde=True, bins=30, ax=axes[1, 1])
axes[1, 1].set_title("Log(Fare) Distribution")

sns.barplot(data=train, x="FamilySize", y="Survived", ax=axes[1, 2])
axes[1, 2].set_title("Survival Rate by Family Size")

sns.heatmap(train.pivot_table(values="Survived", index="Sex", columns="Pclass", aggfunc="mean"),
            annot=True, fmt=".2f", cmap="RdYlGn", ax=axes[1, 3])
axes[1, 3].set_title("Survival Rate: Sex x Class")

plt.tight_layout()
plt.show()

corr_df = train[["Survived", "Pclass", "Age", "SibSp", "Parch", "LogFare", "FamilySize",
                 "IsAlone", "HasCabin", "TicketGroupSize"]].copy()
corr_df["Female"] = (train["Sex"] == "female").astype(int)
plt.figure(figsize=(10, 8))
sns.heatmap(corr_df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()


# ============================================================
# 4. MODEL DEFINITIONS
# ============================================================
def build_preprocess():
    return ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore",
                                                   sparse_output=False))]),
         categorical_features),
    ])


def make_pipe(clf):
    return Pipeline([("prep", build_preprocess()), ("clf", clf)])


def base_estimators():
    return [
        ("lr", LogisticRegression(max_iter=2000, C=0.5)),
        ("svm", SVC(C=1.0, probability=True, random_state=SEED)),
        ("rf", RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=2,
                                      random_state=SEED, n_jobs=-1)),
        ("gb", GradientBoostingClassifier(n_estimators=150, learning_rate=0.05,
                                          max_depth=3, random_state=SEED)),
    ]


classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=2000, C=0.5),
    "KNN (k=15)": KNeighborsClassifier(n_neighbors=15),
    "SVM (RBF)": SVC(C=1.0, probability=True, random_state=SEED),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=500, max_depth=6, min_samples_leaf=2,
                                            random_state=SEED, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=500, max_depth=7, min_samples_leaf=2,
                                        random_state=SEED, n_jobs=-1),
    "AdaBoost": AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=SEED),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=150, learning_rate=0.05,
                                                    max_depth=3, random_state=SEED),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                                             max_iter=200, random_state=SEED),
    "Voting (soft)": VotingClassifier(base_estimators(), voting="soft"),
    "Stacking": StackingClassifier(base_estimators(),
                                   final_estimator=LogisticRegression(max_iter=2000), cv=5),
}

# Optional libraries (used only if installed)
try:
    from xgboost import XGBClassifier
    classifiers["XGBoost"] = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                                           subsample=0.8, colsample_bytree=0.8,
                                           eval_metric="logloss", random_state=SEED, n_jobs=-1)
except ImportError:
    print("xgboost not installed - skipping")
try:
    from lightgbm import LGBMClassifier
    classifiers["LightGBM"] = LGBMClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                                             subsample=0.8, colsample_bytree=0.8,
                                             random_state=SEED, verbose=-1, n_jobs=-1)
except ImportError:
    print("lightgbm not installed - skipping")

pipes = {name: make_pipe(clf) for name, clf in classifiers.items()}

# ============================================================
# 5. HYPERPARAMETER TUNING (Random Forest + Gradient Boosting)
# ============================================================
print("\n" + "=" * 60)
print("HYPERPARAMETER TUNING (RandomizedSearchCV, 30 iterations each)")
print("=" * 60)

search_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

searches = {
    "Random Forest (tuned)": (
        make_pipe(RandomForestClassifier(random_state=SEED, n_jobs=-1)),
        {"clf__n_estimators": [200, 300, 500, 800],
         "clf__max_depth": [3, 4, 5, 6, 8, None],
         "clf__min_samples_leaf": [1, 2, 3, 5, 8],
         "clf__min_samples_split": [2, 5, 10],
         "clf__max_features": ["sqrt", "log2", 0.5]},
    ),
    "Gradient Boosting (tuned)": (
        make_pipe(GradientBoostingClassifier(random_state=SEED)),
        {"clf__n_estimators": [100, 200, 300, 500],
         "clf__learning_rate": [0.01, 0.03, 0.05, 0.1],
         "clf__max_depth": [2, 3, 4],
         "clf__min_samples_leaf": [1, 3, 5, 10],
         "clf__subsample": [0.6, 0.8, 1.0]},
    ),
}

for name, (pipe, grid) in searches.items():
    rs = RandomizedSearchCV(pipe, grid, n_iter=30, cv=search_cv, scoring="accuracy",
                            n_jobs=-1, random_state=SEED)
    rs.fit(X, y)
    pipes[name] = rs.best_estimator_
    print(f"\n{name}")
    print(f"  Best search CV accuracy: {rs.best_score_:.4f}")
    for k, v in rs.best_params_.items():
        print(f"  {k.replace('clf__', '')}: {v}")

# ============================================================
# 6. CROSS-VALIDATED COMPARISON OF ALL MODELS
# ============================================================
print("\n" + "=" * 60)
print("MODEL COMPARISON (5-fold x 3 repeats, accuracy)")
print("=" * 60)

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
rows = []
for name, pipe in pipes.items():
    s = cross_val_score(pipe, X, y, cv=rcv, scoring="accuracy", n_jobs=-1)
    rows.append({"Model": name, "Mean": s.mean(), "Std": s.std(), "Min": s.min(), "Max": s.max()})
    print(f"  done: {name:<28} {s.mean():.4f}")

results = pd.DataFrame(rows).sort_values("Mean", ascending=False).reset_index(drop=True)
results.index += 1
print("\n" + results.round(4).to_string())
print("\nNote: tuned models' scores are slightly optimistic because the")
print("hyperparameters were selected on the same data.")

plt.figure(figsize=(10, 7))
plot_df = results.iloc[::-1]
plt.barh(plot_df["Model"], plot_df["Mean"], xerr=plot_df["Std"], color="steelblue", capsize=3)
plt.xlim(plot_df["Mean"].min() - 0.03, plot_df["Mean"].max() + 0.02)
plt.xlabel("CV Accuracy")
plt.title("Model Comparison")
plt.tight_layout()
plt.show()

best_name = results.loc[1, "Model"]
best_pipe = pipes[best_name]
top5 = results["Model"].head(5).tolist()
print(f"\nBest model by CV: {best_name} ({results.loc[1, 'Mean']:.4f})")

# ============================================================
# 7. ROC CURVES (top 5, out-of-fold probabilities)
# ============================================================
print("\n" + "=" * 60)
print("ROC AUC (top 5 models, out-of-fold)")
print("=" * 60)

plt.figure(figsize=(8, 7))
for name in top5:
    proba = cross_val_predict(pipes[name], X, y, cv=search_cv, method="predict_proba", n_jobs=-1)[:, 1]
    fpr, tpr, _ = roc_curve(y, proba)
    roc_auc = auc(fpr, tpr)
    print(f"  {name:<28} AUC = {roc_auc:.4f}")
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - Top 5 Models")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# ============================================================
# 8. CONFUSION MATRIX + CLASSIFICATION REPORT (best model)
# ============================================================
oof_pred = cross_val_predict(best_pipe, X, y, cv=search_cv, n_jobs=-1)
print("\n" + "=" * 60)
print(f"CLASSIFICATION REPORT - {best_name} (out-of-fold)")
print("=" * 60)
print(classification_report(y, oof_pred, target_names=["Died", "Survived"], digits=3))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y, oof_pred, display_labels=["Died", "Survived"],
                                        cmap="Blues", ax=ax)
ax.set_title(f"Confusion Matrix - {best_name}")
plt.tight_layout()
plt.show()

# ============================================================
# 9. LEARNING CURVE (best model)
# ============================================================
sizes, train_sc, val_sc = learning_curve(best_pipe, X, y, cv=search_cv, scoring="accuracy",
                                         train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1,
                                         random_state=SEED)
plt.figure(figsize=(8, 6))
plt.plot(sizes, train_sc.mean(axis=1), "o-", label="Training accuracy")
plt.plot(sizes, val_sc.mean(axis=1), "o-", label="Validation accuracy")
plt.fill_between(sizes, val_sc.mean(axis=1) - val_sc.std(axis=1),
                 val_sc.mean(axis=1) + val_sc.std(axis=1), alpha=0.15)
plt.xlabel("Training set size")
plt.ylabel("Accuracy")
plt.title(f"Learning Curve - {best_name}")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 10. FEATURE IMPORTANCES (tuned Random Forest)
# ============================================================
rf_pipe = pipes["Random Forest (tuned)"]
names = [n.split("__", 1)[1] for n in rf_pipe.named_steps["prep"].get_feature_names_out()]
imp = pd.Series(rf_pipe.named_steps["clf"].feature_importances_, index=names).sort_values(ascending=False)

print("\n" + "=" * 60)
print("TOP 15 FEATURE IMPORTANCES (tuned Random Forest)")
print("=" * 60)
print(imp.head(15).round(4).to_string())

plt.figure(figsize=(9, 7))
imp.head(15).iloc[::-1].plot(kind="barh", color="darkorange")
plt.xlabel("Importance")
plt.title("Top 15 Feature Importances - Tuned Random Forest")
plt.tight_layout()
plt.show()

# ============================================================
# 11. FIT ALL MODELS ON FULL TRAIN, COMPARE TEST PREDICTIONS
# ============================================================
test_preds = {}
for name, pipe in pipes.items():
    pipe.fit(X, y)
    test_preds[name] = pipe.predict(X_test).astype(int)

preds_df = pd.DataFrame(test_preds)
P = preds_df.values
agree = pd.DataFrame((P[:, :, None] == P[:, None, :]).mean(axis=0),
                     index=preds_df.columns, columns=preds_df.columns)

plt.figure(figsize=(11, 9))
sns.heatmap(agree, annot=True, fmt=".2f", cmap="viridis", vmin=agree.values.min())
plt.title("Agreement Between Models' Test Predictions")
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("PREDICTED SURVIVAL RATE ON TEST SET, BY MODEL")
print("=" * 60)
print(preds_df.mean().sort_values(ascending=False).map("{:.1%}".format).to_string())

# ============================================================
# 12. FINAL SUBMISSION (best CV model)
# ============================================================
final_preds = test_preds[best_name]
submission = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": final_preds})
merged = submission.merge(baseline, on="PassengerId", suffixes=("_model", "_baseline"))
base_agree = (merged["Survived_model"] == merged["Survived_baseline"]).mean()

print("\n" + "=" * 60)
print(f"FINAL SUBMISSION - {best_name}")
print("=" * 60)
print(f"Shape: {submission.shape}  (expected: (418, 2))")
print(f"Predicted survived: {final_preds.sum()} ({final_preds.mean():.1%})")
print(f"Predicted deceased: {(final_preds == 0).sum()} ({1 - final_preds.mean():.1%})")
print(f"Agreement with gender_submission baseline: {base_agree:.1%}")
print("\nFirst 10 predictions:")
print(submission.head(10).to_string(index=False))

submission.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved submission to {OUTPUT_PATH}")

In [ ]:
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (RepeatedStratifiedKFold, StratifiedKFold,
                                     cross_val_score, cross_val_predict,
                                     RandomizedSearchCV, learning_curve)
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_curve, auc)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              AdaBoostClassifier, GradientBoostingClassifier,
                              HistGradientBoostingClassifier,
                              VotingClassifier, StackingClassifier)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
SEED = 42


def header(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def text_bar(value, lo, hi, width=30):
    frac = 0 if hi == lo else (value - lo) / (hi - lo)
    return "#" * max(1, int(round(frac * width)))


# ============================================================
# 1. LOAD DATA
# ============================================================
DATA_DIR = "/kaggle/input/competitions/titanic"
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
baseline = pd.read_csv(f"{DATA_DIR}/gender_submission.csv")

# ============================================================
# 2. FEATURE ENGINEERING
# ============================================================
ticket_counts = pd.concat([train["Ticket"], test["Ticket"]]).value_counts()


def add_features(df):
    df = df.copy()
    df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
    rare = ["Dr", "Rev", "Col", "Major", "Capt", "Sir", "Lady", "Don",
            "Dona", "Jonkheer", "the Countess"]
    df["Title"] = df["Title"].replace(rare, "Rare")
    df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["HasCabin"] = df["Cabin"].notna().astype(int)
    df["Deck"] = df["Cabin"].str[0].fillna("U")
    df["TicketGroupSize"] = df["Ticket"].map(ticket_counts)
    df["LogFare"] = np.log1p(df["Fare"])
    return df


train = add_features(train)
test = add_features(test)

numeric_features = ["Age", "LogFare", "SibSp", "Parch", "FamilySize",
                    "IsAlone", "HasCabin", "TicketGroupSize"]
categorical_features = ["Pclass", "Sex", "Embarked", "Title", "Deck"]

X = train[numeric_features + categorical_features]
y = train["Survived"]
X_test = test[numeric_features + categorical_features]

# ============================================================
# 3. EDA FIGURES AS TABLES
# ============================================================
header("DATA OVERVIEW")
print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Overall survival rate: {y.mean():.1%}")
print("\nMissing values (train):")
print(train.isna().sum()[lambda s: s > 0].to_string())


def survival_table(col, title):
    t = train.groupby(col)["Survived"].agg(Total="count", Survived="sum", Rate="mean")
    t["Died"] = t["Total"] - t["Survived"]
    t = t[["Total", "Died", "Survived", "Rate"]]
    t["Rate"] = t["Rate"].round(3)
    t["Chart"] = [text_bar(r, 0, 1, 25) for r in t["Rate"]]
    print(f"\n--- Survival by {title} ---")
    print(t.to_string())


header("EDA: SURVIVAL BREAKDOWNS (replaces the count plots / bar plots)")
survival_table("Sex", "Sex")
survival_table("Pclass", "Class")
survival_table("Embarked", "Port")
survival_table("Title", "Title")
survival_table("FamilySize", "Family Size")
survival_table("Deck", "Deck")

print("\n--- Survival rate: Sex x Class (replaces the heatmap) ---")
print(train.pivot_table(values="Survived", index="Sex", columns="Pclass",
                        aggfunc="mean").round(3).to_string())
print("\n--- Passenger counts: Sex x Class ---")
print(train.pivot_table(values="Survived", index="Sex", columns="Pclass",
                        aggfunc="count").to_string())

header("EDA: AGE DISTRIBUTION (replaces the age histogram)")
age_bins = [0, 5, 12, 18, 25, 35, 45, 55, 65, 100]
age_cut = pd.cut(train["Age"], bins=age_bins)
age_t = train.groupby(age_cut, observed=True)["Survived"].agg(Total="count", Rate="mean")
age_t["Rate"] = age_t["Rate"].round(3)
age_t["Chart"] = [text_bar(r, 0, 1, 25) for r in age_t["Rate"]]
print(age_t.to_string())
print(f"\nAge summary (survived vs died):")
print(train.groupby("Survived")["Age"].describe().round(2).to_string())

header("EDA: FARE DISTRIBUTION (replaces the log-fare histogram)")
fare_cut = pd.qcut(train["Fare"], q=8, duplicates="drop")
fare_t = train.groupby(fare_cut, observed=True)["Survived"].agg(Total="count", Rate="mean")
fare_t["Rate"] = fare_t["Rate"].round(3)
fare_t["Chart"] = [text_bar(r, 0, 1, 25) for r in fare_t["Rate"]]
print(fare_t.to_string())

header("CORRELATION MATRIX (replaces the correlation heatmap)")
corr_df = train[["Survived", "Pclass", "Age", "SibSp", "Parch", "LogFare", "FamilySize",
                 "IsAlone", "HasCabin", "TicketGroupSize"]].copy()
corr_df["Female"] = (train["Sex"] == "female").astype(int)
print(corr_df.corr().round(2).to_string())
print("\nCorrelation with Survived (sorted):")
print(corr_df.corr()["Survived"].drop("Survived").sort_values(ascending=False).round(3).to_string())


# ============================================================
# 4. MODEL DEFINITIONS
# ============================================================
def build_preprocess():
    return ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), numeric_features),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("onehot", OneHotEncoder(handle_unknown="ignore",
                                                   sparse_output=False))]),
         categorical_features),
    ])


def make_pipe(clf):
    return Pipeline([("prep", build_preprocess()), ("clf", clf)])


def base_estimators():
    return [
        ("lr", LogisticRegression(max_iter=2000, C=0.5)),
        ("svm", SVC(C=1.0, probability=True, random_state=SEED)),
        ("rf", RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=2,
                                      random_state=SEED, n_jobs=-1)),
        ("gb", GradientBoostingClassifier(n_estimators=150, learning_rate=0.05,
                                          max_depth=3, random_state=SEED)),
    ]


classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=2000, C=0.5),
    "KNN (k=15)": KNeighborsClassifier(n_neighbors=15),
    "SVM (RBF)": SVC(C=1.0, probability=True, random_state=SEED),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=500, max_depth=6, min_samples_leaf=2,
                                            random_state=SEED, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=500, max_depth=7, min_samples_leaf=2,
                                        random_state=SEED, n_jobs=-1),
    "AdaBoost": AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=SEED),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=150, learning_rate=0.05,
                                                    max_depth=3, random_state=SEED),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                                             max_iter=200, random_state=SEED),
    "Voting (soft)": VotingClassifier(base_estimators(), voting="soft"),
    "Stacking": StackingClassifier(base_estimators(),
                                   final_estimator=LogisticRegression(max_iter=2000), cv=5),
}

try:
    from xgboost import XGBClassifier
    classifiers["XGBoost"] = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                                           subsample=0.8, colsample_bytree=0.8,
                                           eval_metric="logloss", random_state=SEED, n_jobs=-1)
except ImportError:
    print("xgboost not installed - skipping")
try:
    from lightgbm import LGBMClassifier
    classifiers["LightGBM"] = LGBMClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                                             subsample=0.8, colsample_bytree=0.8,
                                             random_state=SEED, verbose=-1, n_jobs=-1)
except ImportError:
    print("lightgbm not installed - skipping")

pipes = {name: make_pipe(clf) for name, clf in classifiers.items()}

# ============================================================
# 5. HYPERPARAMETER TUNING
# ============================================================
header("HYPERPARAMETER TUNING (RandomizedSearchCV, 30 iterations each)")
search_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

searches = {
    "Random Forest (tuned)": (
        make_pipe(RandomForestClassifier(random_state=SEED, n_jobs=-1)),
        {"clf__n_estimators": [200, 300, 500, 800],
         "clf__max_depth": [3, 4, 5, 6, 8, None],
         "clf__min_samples_leaf": [1, 2, 3, 5, 8],
         "clf__min_samples_split": [2, 5, 10],
         "clf__max_features": ["sqrt", "log2", 0.5]},
    ),
    "Gradient Boosting (tuned)": (
        make_pipe(GradientBoostingClassifier(random_state=SEED)),
        {"clf__n_estimators": [100, 200, 300, 500],
         "clf__learning_rate": [0.01, 0.03, 0.05, 0.1],
         "clf__max_depth": [2, 3, 4],
         "clf__min_samples_leaf": [1, 3, 5, 10],
         "clf__subsample": [0.6, 0.8, 1.0]},
    ),
}

for name, (pipe, grid) in searches.items():
    rs = RandomizedSearchCV(pipe, grid, n_iter=30, cv=search_cv, scoring="accuracy",
                            n_jobs=-1, random_state=SEED)
    rs.fit(X, y)
    pipes[name] = rs.best_estimator_
    print(f"\n{name}  (best search CV accuracy: {rs.best_score_:.4f})")
    for k, v in rs.best_params_.items():
        print(f"  {k.replace('clf__', '')}: {v}")

# ============================================================
# 6. MODEL COMPARISON (text bar chart)
# ============================================================
header("MODEL COMPARISON (5-fold x 3 repeats, accuracy)")
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
rows = []
for name, pipe in pipes.items():
    s = cross_val_score(pipe, X, y, cv=rcv, scoring="accuracy", n_jobs=-1)
    rows.append({"Model": name, "Mean": s.mean(), "Std": s.std(), "Min": s.min(), "Max": s.max()})

results = pd.DataFrame(rows).sort_values("Mean", ascending=False).reset_index(drop=True)
results.index += 1
lo, hi = results["Mean"].min() - 0.01, results["Mean"].max()
results["Chart"] = [text_bar(m, lo, hi, 30) for m in results["Mean"]]
print(results.round(4).to_string())
print("\nNote: tuned models' scores are slightly optimistic because the")
print("hyperparameters were selected on the same data.")

best_name = results.loc[1, "Model"]
best_pipe = pipes[best_name]
top5 = results["Model"].head(5).tolist()
print(f"\nBest model by CV: {best_name} ({results.loc[1, 'Mean']:.4f})")

# ============================================================
# 7. ROC CURVES AS NUMBERS
# ============================================================
header("ROC ANALYSIS - TOP 5 MODELS (out-of-fold)")
fpr_grid = np.array([0.05, 0.10, 0.20, 0.30, 0.40, 0.50])
roc_rows = []
for name in top5:
    proba = cross_val_predict(pipes[name], X, y, cv=search_cv,
                              method="predict_proba", n_jobs=-1)[:, 1]
    fpr, tpr, _ = roc_curve(y, proba)
    row = {"Model": name, "AUC": auc(fpr, tpr)}
    for f, t in zip(fpr_grid, np.interp(fpr_grid, fpr, tpr)):
        row[f"TPR@FPR={f:.2f}"] = t
    roc_rows.append(row)
print(pd.DataFrame(roc_rows).set_index("Model").round(3).to_string())
print("\n(TPR@FPR = share of survivors correctly found when only that share")
print(" of non-survivors is wrongly flagged. Higher is better.)")

# ============================================================
# 8. CONFUSION MATRIX + CLASSIFICATION REPORT
# ============================================================
oof_pred = cross_val_predict(best_pipe, X, y, cv=search_cv, n_jobs=-1)
header(f"CONFUSION MATRIX - {best_name} (out-of-fold)")
cm = confusion_matrix(y, oof_pred)
cm_df = pd.DataFrame(cm, index=["Actual Died", "Actual Survived"],
                     columns=["Pred Died", "Pred Survived"])
print(cm_df.to_string())
print("\nRow-normalised (share of each actual class):")
print(pd.DataFrame(cm / cm.sum(axis=1, keepdims=True), index=cm_df.index,
                   columns=cm_df.columns).round(3).to_string())
print(f"\nOverall accuracy: {(oof_pred == y).mean():.4f}")

header(f"CLASSIFICATION REPORT - {best_name} (out-of-fold)")
print(classification_report(y, oof_pred, target_names=["Died", "Survived"], digits=3))

# ============================================================
# 9. LEARNING CURVE AS A TABLE
# ============================================================
header(f"LEARNING CURVE - {best_name}")
sizes, train_sc, val_sc = learning_curve(best_pipe, X, y, cv=search_cv, scoring="accuracy",
                                         train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1,
                                         shuffle=True, random_state=SEED)
lc = pd.DataFrame({
    "Train size": sizes,
    "Train acc": train_sc.mean(axis=1),
    "Val acc": val_sc.mean(axis=1),
    "Val std": val_sc.std(axis=1),
})
lc["Gap"] = lc["Train acc"] - lc["Val acc"]
lc["Val chart"] = [text_bar(v, 0.6, 0.9, 30) for v in lc["Val acc"]]
print(lc.round(4).to_string(index=False))
print("\n(Gap = train minus validation accuracy; a large gap suggests overfitting.)")

# ============================================================
# 10. FEATURE IMPORTANCES (text bar chart)
# ============================================================
rf_pipe = pipes["Random Forest (tuned)"]
names = [n.split("__", 1)[1] for n in rf_pipe.named_steps["prep"].get_feature_names_out()]
imp = pd.Series(rf_pipe.named_steps["clf"].feature_importances_,
                index=names).sort_values(ascending=False).head(15)

header("TOP 15 FEATURE IMPORTANCES (tuned Random Forest)")
for feat, val in imp.items():
    print(f"{feat:<22} {val:.4f}  {text_bar(val, 0, imp.max(), 35)}")

# ============================================================
# 11. MODEL AGREEMENT + PREDICTED SURVIVAL RATES
# ============================================================
test_preds = {}
for name, pipe in pipes.items():
    pipe.fit(X, y)
    test_preds[name] = pipe.predict(X_test).astype(int)

preds_df = pd.DataFrame(test_preds)
P = preds_df.values
agree = pd.DataFrame((P[:, :, None] == P[:, None, :]).mean(axis=0),
                     index=preds_df.columns, columns=preds_df.columns)

header("AGREEMENT BETWEEN MODELS' TEST PREDICTIONS (replaces the heatmap)")
short = {c: c[:9] for c in agree.columns}
print(agree.rename(columns=short).round(2).to_string())

header("PREDICTED SURVIVAL RATE ON TEST SET, BY MODEL")
rates = preds_df.mean().sort_values(ascending=False)
for name, r in rates.items():
    print(f"{name:<28} {r:.1%}  {text_bar(r, 0, 1, 30)}")

# ============================================================
# 12. FINAL PREDICTION SUMMARY (printed only, no file saved)
# ============================================================
final_preds = test_preds[best_name]
submission = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": final_preds})
merged = submission.merge(baseline, on="PassengerId", suffixes=("_model", "_baseline"))
base_agree = (merged["Survived_model"] == merged["Survived_baseline"]).mean()

header(f"FINAL PREDICTION SUMMARY - {best_name}")
print(f"Shape: {submission.shape}  (expected: (418, 2))")
print(f"Predicted survived: {final_preds.sum()} ({final_preds.mean():.1%})")
print(f"Predicted deceased: {(final_preds == 0).sum()} ({1 - final_preds.mean():.1%})")
print(f"Agreement with gender_submission baseline: {base_agree:.1%}")
print("\nFirst 10 predictions:")
print(submission.head(10).to_string(index=False))
